# Bronze Layer - Data Model Ingestion

This notebook loads the data model from the Excel file at `/Volumes/workspace/default/datamodeling/Data model.xlsx` and ingests all tables into the bronze layer (`workspace.bronze` schema).

In [0]:
# List all sheets in the Excel file
from pyspark.sql.functions import *

file_path = "/Volumes/workspace/default/datamodeling/Data model.xlsx"

# Get sheet names using read_files with listSheets operation
sheets_info = spark.sql(f"""
    SELECT * FROM read_files(
        '{file_path}',
        format => 'excel',
        operation => 'listSheets',
        schemaEvolutionMode => 'none'
    )
""")

print("Available sheets in the Excel file:")
display(sheets_info)

In [0]:
# Read all tables from Excel
print("Reading tables from Excel file...\n")

# Read factSalesTable
fact_sales = spark.sql(f"""
    SELECT * FROM read_files(
        '{file_path}',
        format => 'excel',
        headerRows => 1,
        dataAddress => 'factSalesTable!A1:Z10000',
        schemaEvolutionMode => 'none'
    )
""")

# Read dimension tables
dim_customer = spark.sql(f"""
    SELECT * FROM read_files(
        '{file_path}',
        format => 'excel',
        headerRows => 1,
        dataAddress => 'dimCustomerTable!A1:Z10000',
        schemaEvolutionMode => 'none'
    )
""")

dim_date = spark.sql(f"""
    SELECT * FROM read_files(
        '{file_path}',
        format => 'excel',
        headerRows => 1,
        dataAddress => 'dimDateTable!A1:Z10000',
        schemaEvolutionMode => 'none'
    )
""")

dim_region = spark.sql(f"""
    SELECT * FROM read_files(
        '{file_path}',
        format => 'excel',
        headerRows => 1,
        dataAddress => 'dimRegionTable!A1:Z10000',
        schemaEvolutionMode => 'none'
    )
""")

dim_product = spark.sql(f"""
    SELECT * FROM read_files(
        '{file_path}',
        format => 'excel',
        headerRows => 1,
        dataAddress => 'dimProductTable!A1:Z10000',
        schemaEvolutionMode => 'none'
    )
""")

dim_product_subcat = spark.sql(f"""
    SELECT * FROM read_files(
        '{file_path}',
        format => 'excel',
        headerRows => 1,
        dataAddress => 'dimProductSubcategoryTable!A1:Z10000',
        schemaEvolutionMode => 'none'
    )
""")

dim_product_cat = spark.sql(f"""
    SELECT * FROM read_files(
        '{file_path}',
        format => 'excel',
        headerRows => 1,
        dataAddress => 'dimProductCategoryTable!A1:Z10000',
        schemaEvolutionMode => 'none'
    )
""")

print("✅ All tables loaded successfully")

In [0]:
# Function to clean column names (remove spaces and special characters)
def clean_column_names(df):
    """Remove invalid characters from column names"""
    for column in df.columns:
        new_column = column.replace(" ", "_").replace(",", "").replace(";", "").replace("{", "").replace("}", "").replace("(", "").replace(")", "").replace("\n", "").replace("\t", "").replace("=", "")
        df = df.withColumnRenamed(column, new_column)
    return df

# Clean and prepare factSalesTable
fact_sales_clean = clean_column_names(fact_sales)
fact_sales_clean = fact_sales_clean.select("ProductKey", "OrderDateKey", "CustomerKey", "Gender", "OrderNumber", "OrderQuantity", "List_Price", "Product_Cost")

# Clean and prepare dimension tables
dim_customer_clean = clean_column_names(dim_customer)
dim_customer_clean = dim_customer_clean.select("GeographyKey", "MaritalStatus", "Gender", "CustomerKey", "CustomerName")

dim_date_clean = clean_column_names(dim_date)
dim_date_clean = dim_date_clean.select("DateKey", "FullDateAlternateKey", "EnglishMonthName", "CalendarYear")

dim_region_clean = clean_column_names(dim_region)
dim_region_clean = dim_region_clean.select("GeographyKey", "City", "Region", "Country")

dim_product_clean = clean_column_names(dim_product)
dim_product_clean = dim_product_clean.select("ProductKey", "ProductSubcategoryKey", "Color", "StandardCost", "ListPrice", "Product_Name")

dim_product_subcat_clean = clean_column_names(dim_product_subcat)
dim_product_subcat_clean = dim_product_subcat_clean.select("ProductSubcategoryKey", "ProductSubcategoryName", "ProductCategoryKey")

dim_product_cat_clean = clean_column_names(dim_product_cat)
dim_product_cat_clean = dim_product_cat_clean.filter(col("ProductCategoryKey").isNotNull())
dim_product_cat_clean = dim_product_cat_clean.select("ProductCategoryKey", "ProductCategoryName")

print("✅ Column names cleaned and tables prepared")

In [0]:
# Create bronze schema if it doesn't exist
spark.sql("CREATE SCHEMA IF NOT EXISTS workspace.bronze")

print("Saving tables to workspace.bronze catalog...")
print("=" * 80)

# Save factSalesTable
print("\n1. Saving factSalesTable...")
fact_sales_clean.write.mode("overwrite").saveAsTable("workspace.bronze.fact_sales")
print("   ✓ workspace.bronze.fact_sales created")

# Save dimCustomerTable
print("\n2. Saving dimCustomerTable...")
dim_customer_clean.write.mode("overwrite").saveAsTable("workspace.bronze.dim_customer")
print("   ✓ workspace.bronze.dim_customer created")

# Save dimDateTable
print("\n3. Saving dimDateTable...")
dim_date_clean.write.mode("overwrite").saveAsTable("workspace.bronze.dim_date")
print("   ✓ workspace.bronze.dim_date created")

# Save dimRegionTable
print("\n4. Saving dimRegionTable...")
dim_region_clean.write.mode("overwrite").saveAsTable("workspace.bronze.dim_region")
print("   ✓ workspace.bronze.dim_region created")

# Save dimProductTable
print("\n5. Saving dimProductTable...")
dim_product_clean.write.mode("overwrite").saveAsTable("workspace.bronze.dim_product")
print("   ✓ workspace.bronze.dim_product created")

# Save dimProductSubcategoryTable
print("\n6. Saving dimProductSubcategoryTable...")
dim_product_subcat_clean.write.mode("overwrite").saveAsTable("workspace.bronze.dim_product_subcategory")
print("   ✓ workspace.bronze.dim_product_subcategory created")

# Save dimProductCategoryTable
print("\n7. Saving dimProductCategoryTable...")
dim_product_cat_clean.write.mode("overwrite").saveAsTable("workspace.bronze.dim_product_category")
print("   ✓ workspace.bronze.dim_product_category created")

print("\n" + "=" * 80)
print("✅ All tables successfully saved to workspace.bronze schema!")
print("=" * 80)

In [0]:
# Display summary of all bronze tables
print("\n" + "=" * 80)
print("BRONZE LAYER SUMMARY - workspace.bronze")
print("=" * 80)

tables = spark.sql("SHOW TABLES IN workspace.bronze")
display(tables)

print("\n" + "=" * 80)
print("DATA MODEL STRUCTURE")
print("=" * 80)

# Display table row counts
print("\n📊 FACT TABLE:")
print(f"  • fact_sales: {spark.table('workspace.bronze.fact_sales').count()} rows")

print("\n📋 DIMENSION TABLES:")
print(f"  • dim_customer: {spark.table('workspace.bronze.dim_customer').count()} rows")
print(f"  • dim_date: {spark.table('workspace.bronze.dim_date').count()} rows")
print(f"  • dim_region: {spark.table('workspace.bronze.dim_region').count()} rows")
print(f"  • dim_product: {spark.table('workspace.bronze.dim_product').count()} rows")
print(f"  • dim_product_subcategory: {spark.table('workspace.bronze.dim_product_subcategory').count()} rows")
print(f"  • dim_product_category: {spark.table('workspace.bronze.dim_product_category').count()} rows")

print("\n" + "=" * 80)
print("✅ Bronze layer successfully loaded from Excel file!")
print("=" * 80)